# Email Spam Detection Model

A complete machine learning project using a **synthetic email dataset**. The model combines email subject and body text, converts the text into TF-IDF features, and uses Logistic Regression to classify emails as **Ham (0)** or **Spam (1)**.

**Project type:** Binary text classification  
**Dataset:** `email_spam_synthetic_dataset.csv`  
**Reproducibility:** Random seed = 42


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)


## 1. Load the synthetic dataset

In [ ]:
df = pd.read_csv('email_spam_synthetic_dataset.csv')
print('Shape:', df.shape)
display(df.head())


## 2. Data quality checks

The synthetic dataset was created specifically for this project. We still check duplicates, missing values, and class distribution before training.

In [ ]:
print('Missing values:')
display(df.isna().sum())

print('Duplicate rows:', df.duplicated().sum())
print('\nClass distribution:')
display(df['label'].value_counts().rename(index={0: 'Ham', 1: 'Spam'}))


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label')
plt.xticks([0, 1], ['Ham', 'Spam'])
plt.title('Email Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Emails')
plt.show()


## 3. Prepare the text features

The subject and body are combined into one text field. TF-IDF is used to represent important words and word pairs numerically.

In [ ]:
df['text'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')

X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))


## 4. Build the machine learning pipeline

**TF-IDF Vectorizer:** converts email text into numerical features.  
**Logistic Regression:** learns the relationship between those features and the spam/ham labels.

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2,
        max_features=15000
    )),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Model training completed.')


## 5. Evaluate the model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Score': [accuracy, precision, recall, f1]
})
display(metrics.style.format({'Score': '{:.4f}'}))

print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=['Ham', 'Spam'],
    yticklabels=['Ham', 'Spam']
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()


## 6. Test the model on new emails

The following examples demonstrate how the trained model can classify previously unseen email text.

In [ ]:
new_emails = [
    'Congratulations! You won a cash reward. Click the link now to claim your prize.',
    'Hi team, please review the project report before tomorrow afternoon.',
    'URGENT: verify your account immediately to avoid suspension and claim your bonus.',
    'Your meeting has been moved to 3 PM. Please confirm your availability.'
]

new_predictions = model.predict(new_emails)
new_probabilities = model.predict_proba(new_emails)[:, 1]

results = pd.DataFrame({
    'Email': new_emails,
    'Prediction': np.where(new_predictions == 1, 'Spam', 'Ham'),
    'Spam Probability': new_probabilities
})
display(results.style.format({'Spam Probability': '{:.2%}'}))


## 7. Conclusion

The project demonstrates an end-to-end NLP classification workflow: synthetic data creation, data validation, text preprocessing, TF-IDF feature extraction, Logistic Regression training, evaluation, and prediction on new emails.

Because the dataset is synthetic, the reported performance should not be interpreted as production-level spam detection performance. A real deployment would require a large, diverse, continuously updated corpus of real emails and careful handling of false positives.